In [57]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

In [58]:
df = pd.read_csv("../data/silver/my1_cleaned.csv", parse_dates=['datetime'], index_col='datetime')
df.head()

,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m
datetime,,,,,,,,,,,
2018-07-24 00:00:00,20.35614,13.2,2402,0.0,0.0,20.7,71.4,6.0,240.0,1014.9,229.4208
2018-07-24 14:00:00,19.10883,22.7,3644,0.0,963.0,28.7,42.3,9.0,240.0,1013.2,1798.7944
2018-07-24 15:00:00,16.91356,17.8,3672,0.0,1898.0,28.8,43.6,6.0,250.0,1013.2,1731.4312
2018-07-24 16:00:00,18.01119,19.8,3507,0.0,1304.0,29.3,42.5,8.0,240.0,1012.9,1655.7957
2018-07-24 17:00:00,19.00904,19.0,3707,0.0,1199.0,29.1,40.0,8.0,250.0,1012.9,1563.8152


## Adding temporal variables

Unix time, julian day, day of week, hour of day

In [59]:
df['unix_time'] = df.index.astype(np.int64) // 10**9
df['julian_day'] = df.index.dayofyear
df['day_of_week'] = df.index.dayofweek
df['hour_of_day'] = df.index.hour

## Transforming variables
Decomposing wind and transforming units.

In [60]:
# winds speed from knots to m/s
df['wind_speed'] = df['wind_speed'] * 0.514444

wd_rad = np.deg2rad(df['wind_direction'])
df['wind_u'] = -df['wind_speed'] * np.sin(wd_rad) # + = east
df['wind_v'] = -df['wind_speed'] * np.cos(wd_rad) # + = north

# drop
df.drop(columns=['wind_speed', 'wind_direction'], inplace=True)
df.head()

,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day,wind_u,wind_v
datetime,,,,,,,,,,,,,,,
2018-07-24 00:00:00,20.35614,13.2,2402,0.0,0.0,20.7,71.4,1014.9,229.4208,1532390,205,1,0,2.673129,1.543332
2018-07-24 14:00:00,19.10883,22.7,3644,0.0,963.0,28.7,42.3,1013.2,1798.7944,1532440,205,1,14,4.009694,2.314998
2018-07-24 15:00:00,16.91356,17.8,3672,0.0,1898.0,28.8,43.6,1013.2,1731.4312,1532444,205,1,15,2.900515,1.055701
2018-07-24 16:00:00,18.01119,19.8,3507,0.0,1304.0,29.3,42.5,1012.9,1655.7957,1532448,205,1,16,3.564173,2.057776
2018-07-24 17:00:00,19.00904,19.0,3707,0.0,1199.0,29.1,40.0,1012.9,1563.8152,1532451,205,1,17,3.867354,1.407602


# Creating hourly datasets

Hourly datasets with raw variables

In [61]:
# remove index
hourly_o3 = df.reset_index().drop(columns=['datetime', 'pm25'])
hourly_pm25 = df.reset_index().drop(columns=['datetime', 'ozone'])

display(hourly_o3.head())
display(hourly_pm25.head())

,ozone,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day,wind_u,wind_v
0,20.35614,2402,0.0,0.0,20.7,71.4,1014.9,229.4208,1532390,205,1,0,2.673129,1.543332
1,19.10883,3644,0.0,963.0,28.7,42.3,1013.2,1798.7944,1532440,205,1,14,4.009694,2.314998
2,16.91356,3672,0.0,1898.0,28.8,43.6,1013.2,1731.4312,1532444,205,1,15,2.900515,1.055701
3,18.01119,3507,0.0,1304.0,29.3,42.5,1012.9,1655.7957,1532448,205,1,16,3.564173,2.057776
4,19.00904,3707,0.0,1199.0,29.1,40.0,1012.9,1563.8152,1532451,205,1,17,3.867354,1.407602


,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day,wind_u,wind_v
0,13.2,2402,0.0,0.0,20.7,71.4,1014.9,229.4208,1532390,205,1,0,2.673129,1.543332
1,22.7,3644,0.0,963.0,28.7,42.3,1013.2,1798.7944,1532440,205,1,14,4.009694,2.314998
2,17.8,3672,0.0,1898.0,28.8,43.6,1013.2,1731.4312,1532444,205,1,15,2.900515,1.055701
3,19.8,3507,0.0,1304.0,29.3,42.5,1012.9,1655.7957,1532448,205,1,16,3.564173,2.057776
4,19.0,3707,0.0,1199.0,29.1,40.0,1012.9,1563.8152,1532451,205,1,17,3.867354,1.407602


## Creating transformed, daily datasets

Daily means and transformed variables - easier for SR

In [62]:
# removing hour of day, and unix time (we will rebuild that as a linear time varaible)
# removing "categorical" variables (we will rebuild)
daily_df = df.drop(columns=['hour_of_day', 'unix_time', 'julian_day', 'day_of_week'])

print("Size before:", len(daily_df))

# creating means
daily_df['date'] = daily_df.index.date

daily_df = daily_df.reset_index(drop=True)

daily_df = daily_df.groupby('date').mean().reset_index()

print("Size after:", len(daily_df))

# adding a linear time variable | t = 0 for the first day, t = 1 for the last day

# guarantee that the data is sorted by date
daily_df = daily_df.sort_values(by='date').reset_index(drop=True)

daily_df['t'] = np.linspace(0, 1, len(daily_df))

# readd julian day
daily_df['julian_day'] = pd.to_datetime(daily_df['date']).dt.dayofyear

# building jsin and jcos variables: 2pij/365
daily_df['jcos'] = np.cos(2 * np.pi * daily_df['julian_day'] / 365)
daily_df['jsin'] = np.sin(2 * np.pi * daily_df['julian_day'] / 365)

# jsum = jsin + jcos => highs on winter, lows on summer
daily_df['jsum'] = daily_df['jsin'] + daily_df['jcos']

# dropping aux
daily_df.drop(columns=['julian_day', 'jcos', 'jsin'], inplace=True)

# readding day of week
daily_df['day_of_week'] = pd.to_datetime(daily_df['date']).dt.dayofweek

# creating dw_wave: np.cos(1.4 * np.pi * df['dw'] / 14) + np.sin(1.2 * np.pi * df['dw'] / 7)
daily_df['dw_wave'] = np.cos(1.4 * np.pi * daily_df['day_of_week'] / 14) + np.sin(1.2 * np.pi * daily_df['day_of_week'] / 7)

# drop dw
daily_df.drop(columns=['day_of_week'], inplace=True)

daily_df.head()

Size before: 31406
Size after: 1329


,date,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,wind_u,wind_v,t,jsum,dw_wave
0,2018-07-24,19.545388,20.550000,3519.166667,0.000000,500.416667,26.316667,49.258333,1013.733333,842.106783,2.875906,1.583041,0.000000,-1.303633,1.463956
1,2018-07-25,21.079221,16.086957,3119.913043,0.000000,914.913043,24.604348,56.413043,1015.791304,756.230172,0.469973,-0.445010,0.000753,-1.312876,1.689613
2,2018-07-26,18.423767,28.023077,3278.730769,0.000000,881.769231,26.934615,55.396154,1012.800000,743.703681,0.340187,1.076987,0.001506,-1.321731,1.586778
3,2018-07-27,20.871057,20.876923,3234.307692,0.476923,789.230769,24.165385,61.326923,1009.053846,506.096748,1.724387,0.645517,0.002259,-1.330194,1.143590
4,2018-07-28,25.548159,11.261538,3243.769231,0.015385,886.923077,19.511538,59.757692,1006.434615,1288.799575,4.118412,2.682399,0.003012,-1.338262,0.433884


In [63]:
# building pollutant specific dataframes
daily_o3 = daily_df.drop(columns=['date', 'pm25'])
daily_pm25 = daily_df.drop(columns=['date', 'ozone'])

display(daily_o3.head())
display(daily_pm25.head())

,ozone,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,wind_u,wind_v,t,jsum,dw_wave
0,19.545388,3519.166667,0.000000,500.416667,26.316667,49.258333,1013.733333,842.106783,2.875906,1.583041,0.000000,-1.303633,1.463956
1,21.079221,3119.913043,0.000000,914.913043,24.604348,56.413043,1015.791304,756.230172,0.469973,-0.445010,0.000753,-1.312876,1.689613
2,18.423767,3278.730769,0.000000,881.769231,26.934615,55.396154,1012.800000,743.703681,0.340187,1.076987,0.001506,-1.321731,1.586778
3,20.871057,3234.307692,0.476923,789.230769,24.165385,61.326923,1009.053846,506.096748,1.724387,0.645517,0.002259,-1.330194,1.143590
4,25.548159,3243.769231,0.015385,886.923077,19.511538,59.757692,1006.434615,1288.799575,4.118412,2.682399,0.003012,-1.338262,0.433884


,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,wind_u,wind_v,t,jsum,dw_wave
0,20.550000,3519.166667,0.000000,500.416667,26.316667,49.258333,1013.733333,842.106783,2.875906,1.583041,0.000000,-1.303633,1.463956
1,16.086957,3119.913043,0.000000,914.913043,24.604348,56.413043,1015.791304,756.230172,0.469973,-0.445010,0.000753,-1.312876,1.689613
2,28.023077,3278.730769,0.000000,881.769231,26.934615,55.396154,1012.800000,743.703681,0.340187,1.076987,0.001506,-1.321731,1.586778
3,20.876923,3234.307692,0.476923,789.230769,24.165385,61.326923,1009.053846,506.096748,1.724387,0.645517,0.002259,-1.330194,1.143590
4,11.261538,3243.769231,0.015385,886.923077,19.511538,59.757692,1006.434615,1288.799575,4.118412,2.682399,0.003012,-1.338262,0.433884


# Saving data

Pollutants renamed to "target"

In [ ]:
hourly_o3_out = hourly_o3.copy().rename(columns={'ozone': 'target'})
hourly_pm25_out = hourly_pm25.copy().rename(columns={'pm25': 'target'})

daily_o3_out = daily_o3.copy().rename(columns={'ozone': 'target'})
daily_pm25_out = daily_pm25.copy().rename(columns={'pm25': 'target'})

# hourly data
hourly_o3_out.to_csv("../data/gold/my1_hourly_o3.csv", index=False)
hourly_pm25_out.to_csv("../data/gold/my1_hourly_pm25.csv", index=False)

# daily data
daily_o3_out.to_csv("../data/gold/my1_daily_o3.csv", index=False)
daily_pm25_out.to_csv("../data/gold/my1_daily_pm25.csv", index=False)